# Swissprot Data

In [1]:
import pandas as pd
import numpy as np
import re

## Downloading 2023-05 Swissprot Data

```
wget https://ftp.uniprot.org/pub/databases/uniprot/previous_releases/release-2023_01/knowledgebase/uniprot_sprot-only2023_01.tar.gz
tar -xvzf uniprot_sprot-only2023_01.tar.gz
pigz -d uniprot_sprot.xml.gz
```

In [2]:
from lxml import etree as ET
import pandas as pd
from tqdm import tqdm

ns = {"ns": "http://uniprot.org/uniprot"}
records = []

context = ET.iterparse("uniprot/uniprot_sprot.xml", events=("end",), tag="{http://uniprot.org/uniprot}entry")

for event, elem in tqdm(context, desc="Parsing UniProt entries", unit=" entry"):
    # Primary accession
    accession = elem.find("ns:accession", namespaces=ns)

    # Function comment
    comment_function = elem.find("ns:comment[@type='function']/ns:text", namespaces=ns)

    # Protein name (take recommended fullName if available)
    protein_name = elem.find("ns:protein/ns:recommendedName/ns:fullName", namespaces=ns)
    if protein_name is None:
        protein_name = elem.find("ns:protein/ns:submittedName/ns:fullName", namespaces=ns)

    # Organism scientific name
    organism = elem.find("ns:organism/ns:name[@type='scientific']", namespaces=ns)

    # Subcellular location(s) → can be multiple
    subcellular_locations = elem.findall("ns:comment[@type='subcellular location']/ns:subcellularLocation/ns:location", namespaces=ns)
    subcellular_locations_text = "; ".join([loc.text for loc in subcellular_locations if loc is not None])

    # Sequence and length
    sequence_elem = elem.find("ns:sequence", namespaces=ns)
    sequence = sequence_elem.text.replace("\n", "") if sequence_elem is not None else None
    length = len(sequence) if sequence is not None else None

    # Store record
    records.append({
        "protein_id": accession.text if accession is not None else None,
        "protein_names": protein_name.text if protein_name is not None else None,
        "protein_function": comment_function.text if comment_function is not None else None,
        "organism": organism.text if organism is not None else None,
        "length": length,
        "subcellular_location": subcellular_locations_text if subcellular_locations_text else None,
        "sequence": sequence
    })

    # Free memory
    elem.clear()

# Convert to dataframe
swissprot_data = pd.DataFrame(records)

Parsing UniProt entries: 569213 entry [02:46, 3419.26 entry/s]


## Removing Proteins already in CAFA5

In [3]:
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

In [4]:
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

all_proteins = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all_proteins = all_proteins['protein_id']
all_proteins.drop_duplicates(inplace=True)
all_proteins.reset_index(drop=True, inplace=True)

In [5]:
all_proteins

0         A0A087X1C5
1         A0A0B4J2F0
2         A0A0A7EPL0
3         A0A0B4J1G0
4         A0A0B4J1N3
             ...    
201752        Q4A3R3
201753        Q5E9L2
201754        Q3T0K9
201755        Q58CX7
201756        A7E300
Name: protein_id, Length: 201757, dtype: object

In [6]:
swissprot_data = swissprot_data[~swissprot_data['protein_id'].isin(all_proteins)]

In [7]:
len(swissprot_data)

417097

## Downloading the older GOA release for GO terms. 2023-01
```
wget https://release.geneontology.org/2023-01-01/annotations/goa_uniprot_all.gaf.gz
pigz -d goa_uniprot_all.gaf.gz
```

In [8]:
swissprot_data['protein_id'].to_csv('swissprot_proteins.txt', index=False,header=False)

```awk -F'\t' 'NR==FNR {ids[$1]; next} $2 in ids' swissprot_proteins.txt goa_uniprot_all.gaf > goa_uniprot_all_subset.gaf```

In [9]:
gaf_cols = [
    "DB",                  # 1. Database (e.g. UniProtKB)
    "DB_Object_ID",        # 2. Accession (e.g. P12345)
    "DB_Object_Symbol",    # 3. Gene symbol
    "Qualifier",           # 4. Relation/qualifier (e.g. NOT, enables)
    "GO_ID",               # 5. GO term (e.g. GO:0003993)
    "Reference",           # 6. Reference(s) supporting annotation
    "Evidence_Code",       # 7. Evidence code (e.g. IMP, IEA)
    "With_From",           # 8. With/From (optional extra evidence)
    "Aspect",              # 9. Aspect: F=Function, P=Process, C=Component
    "DB_Object_Name",      # 10. Full name of gene/protein
    "Synonym",             # 11. Synonyms
    "DB_Object_Type",      # 12. Type (protein, complex, etc.)
    "Taxon",               # 13. Taxon ID(s)
    "Date",                # 14. Annotation date (YYYYMMDD)
    "Assigned_By",         # 15. Submitting database/group
    "Annotation_Ext",      # 16. Annotation extension(s)
    "Gene_Product_Form_ID" # 17. Isoform (optional, e.g. UniProtKB:P12345-2)
]

In [10]:
goa_annotations = pd.read_csv('goa_uniprot_all_subset.gaf',names=gaf_cols, sep='\t')

/tmp/ipykernel_631268/1528332033.py:1: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  goa_annotations = pd.read_csv('goa_uniprot_all_subset.gaf',names=gaf_cols, sep='\t')


In [11]:
swissprot_proteins = pd.read_csv('swissprot_proteins.txt', names=['protein_id'])

In [12]:
swissprot_proteins['protein_id']

0         P0C9F0
1         P0C9F1
2         P0C9F2
3         P0C9E9
4         Q65209
           ...  
417092    P38442
417093    A1YEV9
417094    A1YG26
417095    A2T712
417096    A2T7L7
Name: protein_id, Length: 417097, dtype: object

In [13]:
goa_annotations

,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,Reference,Evidence_Code,With_From,Aspect,DB_Object_Name,Synonym,DB_Object_Type,Taxon,Date,Assigned_By,Annotation_Ext,Gene_Product_Form_ID
0,UniProtKB,B3R0P5,rpmI,enables,GO:0003735,GO_REF:0000002,IEA,InterPro:IPR001706|InterPro:IPR018265|InterPro...,F,50S ribosomal protein L35,rpmI|ATP_00442,protein,taxon:482235,20221109,InterPro,NaN,NaN
1,UniProtKB,B3R0P5,rpmI,located_in,GO:0005840,GO_REF:0000002,IEA,InterPro:IPR001706|InterPro:IPR018265|InterPro...,C,50S ribosomal protein L35,rpmI|ATP_00442,protein,taxon:482235,20221109,InterPro,NaN,NaN
2,UniProtKB,B3R0P5,rpmI,involved_in,GO:0006412,GO_REF:0000002,IEA,InterPro:IPR001706|InterPro:IPR018265|InterPro...,P,50S ribosomal protein L35,rpmI|ATP_00442,protein,taxon:482235,20221109,InterPro,NaN,NaN
3,UniProtKB,B3R0P5,rpmI,part_of,GO:1990904,GO_REF:0000043,IEA,UniProtKB-KW:KW-0687,C,50S ribosomal protein L35,rpmI|ATP_00442,protein,taxon:482235,20221109,UniProt,NaN,NaN
4,UniProtKB,B3R0P5,rpmI,located_in,GO:0005840,GO_REF:0000043,IEA,UniProtKB-KW:KW-0689,C,50S ribosomal protein L35,rpmI|ATP_00442,protein,taxon:482235,20221109,UniProt,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4933905,UniProtKB,B1KR03,tatB,part_of,GO:0033281,GO_REF:0000104,IEA,UniRule:UR000082074,C,Sec-independent protein translocase protein TatB,tatB|Swoo_0523,protein,taxon:392500,20221109,UniProt,NaN,NaN
4933906,UniProtKB,Q01256,arsR,enables,GO:0003700,GO_REF:0000002,IEA,InterPro:IPR001845|InterPro:IPR018334,F,Arsenical resistance operon repressor,arsR,protein,taxon:1288,20221109,InterPro,NaN,NaN
4933907,UniProtKB,Q01256,arsR,involved_in,GO:0006355,GO_REF:0000002,IEA,InterPro:IPR001845|InterPro:IPR018334,P,Arsenical resistance operon repressor,arsR,protein,taxon:1288,20221109,InterPro,NaN,NaN
4933908,UniProtKB,Q01256,arsR,enables,GO:0003677,GO_REF:0000043,IEA,UniProtKB-KW:KW-0238,F,Arsenical resistance operon repressor,arsR,protein,taxon:1288,20221109,UniProt,NaN,NaN


In [14]:
len(goa_annotations[goa_annotations['DB_Object_ID'].isin(swissprot_proteins['protein_id'])])

4933910

In [15]:
goa_annotations['DB_Object_ID'].drop_duplicates()

0          B3R0P5
6          Q9X0G8
17         B2UCF1
34         T2KLZ8
41         O85987
            ...  
4933850    Q5H2E0
4933864    Q5L723
4933877    Q66BN4
4933894    B1KR03
4933906    Q01256
Name: DB_Object_ID, Length: 396578, dtype: object

In [16]:
from tqdm import tqdm
import pandas as pd

def collapse_go_terms_per_protein(df):
    go_fields = ["GO_ID", "Aspect"]

    def agg_go_group(group):
        first = group.iloc[0].to_dict()

        go_ids = group["GO_ID"].tolist()  # Keep original list
        go_bp = group.loc[group["Aspect"] == "P", "GO_ID"].tolist()
        go_mf = group.loc[group["Aspect"] == "F", "GO_ID"].tolist()
        go_cc = group.loc[group["Aspect"] == "C", "GO_ID"].tolist()

        out = {
            "protein_id": first["DB_Object_ID"],
            "go_ids": go_ids,
            "go_bp": go_bp if go_bp else np.nan,
            "go_mf": go_mf if go_mf else np.nan,
            "go_cc": go_cc if go_cc else np.nan,
        }

        # Add other fields from first row
        for col in df.columns:
            if col not in go_fields + ["DB_Object_ID"]:
                out[col] = first[col]

        return pd.Series(out)

    tqdm.pandas(desc="Collapsing GO terms")
    collapsed_df = df.groupby("DB_Object_ID").progress_apply(agg_go_group).reset_index(drop=True)
    return collapsed_df

In [17]:
go_terms = collapse_go_terms_per_protein(goa_annotations)

Collapsing GO terms: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 396578/396578 [05:50<00:00, 1130.15it/s]


In [18]:
swissprot_data = swissprot_data.merge(go_terms[['protein_id','go_ids','go_bp','go_mf','go_cc']], how='left',on='protein_id')
swissprot_data

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
0,P0C9F0,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Pig/Kenya/K...,122,None,MVRLFYNPIKYLFYRRSCKKRLRKALKKLNFYHPPKECCQIYRLLE...,NaN,NaN,NaN,NaN
1,P0C9F1,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Tick/Malawi...,126,None,MVRLFHNPIKCLFYRGSRKTREKKLRKSLKKLNFYHPPGDCCQIYR...,NaN,NaN,NaN,NaN
2,P0C9F2,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Tick/South ...,124,None,MVRLFRNPIKCIFYRRSRKIQEKKLRKSLKKLNFYHPPEDCCQIYR...,NaN,NaN,NaN,NaN
3,P0C9E9,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Warthog/Nam...,124,None,MVRLFRNPIKCIFYRRSRKIQEKKLRKSLKKLNFYHPPEDCCQIYR...,NaN,NaN,NaN,NaN
4,Q65209,Protein MGF 100-2L,"Plays a role in virus cell tropism, and may be...",African swine fever virus (strain Badajoz 1971...,141,None,MGNKESKYLEMCSEEAWLNIPNIFKCIFIRKLFYNKWLKYQEKKLK...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
417092,P38442,Zona occludens toxin,Increases the permeability of the small intest...,Vibrio cholerae serotype O1 (strain ATCC 39315...,399,None,MSIFIHHGAPGSYKTSGALWLRLLPAIKSGRHIITNVRGLNLERMA...,"[GO:0090729, GO:0035821]",[GO:0035821],[GO:0090729],NaN
417093,A1YEV9,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Gorilla gorilla gorilla,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872]","[GO:0005634, GO:0005634]"
417094,A1YG26,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Pan paniscus,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872, GO:0000978, GO:0001228]","[GO:0005634, GO:0005634]"
417095,A2T712,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Pan troglodytes,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872, GO:0000978, GO:0001228]","[GO:0005634, GO:0005634]"


### Testing and QC the Data

In [19]:
swissprot_data[swissprot_data['protein_names'].isna()].shape

(0, 11)

In [20]:
swissprot_data[swissprot_data['protein_function'].isna()].shape

(76300, 11)

In [21]:
swissprot_data[swissprot_data['organism'].isna()].shape

(0, 11)

In [22]:
swissprot_data[swissprot_data['length'].isna()].shape

(0, 11)

In [23]:
swissprot_data[swissprot_data['subcellular_location'].isna()].shape

(178503, 11)

In [24]:
swissprot_data[swissprot_data['sequence'].isna()].shape

(0, 11)

In [25]:
swissprot_data[swissprot_data['go_ids'].isna()].shape

(20519, 11)

Checking the Distribution of protein names and sequence

In [26]:
swissprot_data['protein_names'].value_counts()

protein_names
Cytochrome b                                                       1745
30S ribosomal protein S19                                           866
50S ribosomal protein L2                                            854
DNA ligase                                                          854
50S ribosomal protein L14                                           851
                                                                   ... 
Uncharacterized protein MT0764                                        1
Uncharacterized protein PM0739                                        1
MEMO1 family protein YN1551_0739                                      1
Probable transcriptional regulatory protein SYNAS_07390               1
Uncharacterized 36.3 kDa protein in nrdC-mobD intergenic region       1
Name: count, Length: 65185, dtype: int64

In [27]:
protein_name_sort = swissprot_data['protein_names'].sort_values(key=lambda x: x.str.len(), ascending=False).reset_index(drop=True)

In [28]:
# Checking length of the longest name
len(protein_name_sort[0])

117

In [29]:
swissprot_data['protein_id'].value_counts(ascending=False)

protein_id
P0C9F0    1
Q8S8X6    1
A8Y9D2    1
A4QLQ4    1
Q0G9G2    1
         ..
Q9ZLL0    1
Q1CTP7    1
B5Z6Z7    1
B6JLL3    1
A2T7L7    1
Name: count, Length: 417097, dtype: int64

In [30]:
sequence_sort = swissprot_data['sequence'].sort_values(key=lambda x: x.str.len(), ascending=False).reset_index(drop=True)

In [31]:
len(sequence_sort[0])

15639

Checking if all 3 subprocess go terms add up to go_ids

In [32]:
from tqdm import tqdm
tqdm.pandas(desc="Sanity check: GO categories sum to go_ids")

def check_go_sum(row):
    go_ids = set(row["go_ids"]) if isinstance(row["go_ids"], list) else set()
    go_union = set()
    for col in ["go_bp", "go_mf", "go_cc"]:
        if isinstance(row[col], list):
            go_union.update(row[col])
    return go_ids == go_union

# Apply to all rows
sanity_check_passed = swissprot_data.progress_apply(check_go_sum, axis=1).all()

print(f"Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? {sanity_check_passed}")

# Optional: if you want to see which rows fail
failed_rows = swissprot_data[~swissprot_data.progress_apply(check_go_sum, axis=1)]
print(f"Number of rows failing sanity check: {len(failed_rows)}")

Sanity check: GO categories sum to go_ids: 100%|████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:07<00:00, 58351.12it/s]


Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? True


Sanity check: GO categories sum to go_ids: 100%|████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:07<00:00, 58097.19it/s]

Number of rows failing sanity check: 0


## Checking if all GO terms from GOA are in go-basic

In [33]:
from tqdm import tqdm
from goatools.obo_parser import GODag

# Load ontology
obodag = GODag("go-basic.obo")

# Precompute set of valid GO terms
valid_go_terms = set(obodag.keys())

# Track invalid terms and affected proteins
invalid_terms_set = set()
proteins_with_invalid_terms = set()

def clean_go_terms(go_terms, protein_id):
    """Filter GO terms into valid only; track invalid terms and affected proteins."""
    if not isinstance(go_terms, list) or not go_terms:
        return None
    
    valid = [go for go in go_terms if go in valid_go_terms]
    invalid = [go for go in go_terms if go not in valid_go_terms]
    
    if invalid:
        proteins_with_invalid_terms.add(protein_id)
        invalid_terms_set.update(invalid)
    
    return valid if valid else None

# Columns to clean
go_columns = ["go_ids", "go_bp", "go_mf", "go_cc"]

tqdm.pandas(desc="Cleaning GO terms")

for col in go_columns:
    swissprot_data[col] = swissprot_data.apply(
        lambda row: clean_go_terms(row[col], row.name), axis=1
    )

# Summary
print(f"Unique invalid GO terms across all columns: {len(invalid_terms_set)}")
print(f"Number of unique proteins with at least one invalid GO term: {len(proteins_with_invalid_terms)}")

go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms
Unique invalid GO terms across all columns: 14
Number of unique proteins with at least one invalid GO term: 1289


**Removed these go terms as they were obsolete**

## Adding in all Parent GO Terms

In [34]:
from tqdm import tqdm
from goatools.obo_parser import GODag

# Load ontology
obodag = GODag("go-basic.obo")

# Precompute parent terms for all GO terms
print("Precomputing parent terms...")
go_to_parents = {}
for go_id, term in tqdm(obodag.items(), desc="Building parent map"):
    go_to_parents[go_id] = term.get_all_parents()

def extend_with_parents(go_terms):
    if not isinstance(go_terms, list) or not go_terms:
        return None
    
    extended = set(go_terms)  # start with original terms
    for go_id in go_terms:
        if go_id in go_to_parents:
            extended.update(go_to_parents[go_id])
    return list(extended)

# Apply to dataframe
tqdm.pandas(desc="Extending GO terms with parents")
swissprot_data["go_ids_extended"] = swissprot_data["go_ids"].progress_apply(extend_with_parents)
swissprot_data["go_bp_extended"] = swissprot_data["go_bp"].progress_apply(extend_with_parents)
swissprot_data["go_mf_extended"] = swissprot_data["go_mf"].progress_apply(extend_with_parents)
swissprot_data["go_cc_extended"] = swissprot_data["go_cc"].progress_apply(extend_with_parents)

# Summary: check average size increase
avg_before = swissprot_data["go_ids"].dropna().map(len).mean()
avg_after = swissprot_data["go_ids_extended"].dropna().map(len).mean()
print(f"Average #terms before extension: {avg_before:.2f}")
print(f"Average #terms after extension: {avg_after:.2f}")

go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms
Precomputing parent terms...


Extending GO terms with parents: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:00<00:00, 694946.93it/s]


Average #terms before extension: 12.44
Average #terms after extension: 43.26


In [35]:
swissprot_data

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,go_ids_extended,go_bp_extended,go_mf_extended,go_cc_extended
0,P0C9F0,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Pig/Kenya/K...,122,None,MVRLFYNPIKYLFYRRSCKKRLRKALKKLNFYHPPKECCQIYRLLE...,None,None,None,None,None,None,None,None
1,P0C9F1,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Tick/Malawi...,126,None,MVRLFHNPIKCLFYRGSRKTREKKLRKSLKKLNFYHPPGDCCQIYR...,None,None,None,None,None,None,None,None
2,P0C9F2,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Tick/South ...,124,None,MVRLFRNPIKCIFYRRSRKIQEKKLRKSLKKLNFYHPPEDCCQIYR...,None,None,None,None,None,None,None,None
3,P0C9E9,Protein MGF 100-1R,"Plays a role in virus cell tropism, and may be...",African swine fever virus (isolate Warthog/Nam...,124,None,MVRLFRNPIKCIFYRRSRKIQEKKLRKSLKKLNFYHPPEDCCQIYR...,None,None,None,None,None,None,None,None
4,Q65209,Protein MGF 100-2L,"Plays a role in virus cell tropism, and may be...",African swine fever virus (strain Badajoz 1971...,141,None,MGNKESKYLEMCSEEAWLNIPNIFKCIFIRKLFYNKWLKYQEKKLK...,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417092,P38442,Zona occludens toxin,Increases the permeability of the small intest...,Vibrio cholerae serotype O1 (strain ATCC 39315...,399,None,MSIFIHHGAPGSYKTSGALWLRLLPAIKSGRHIITNVRGLNLERMA...,"[GO:0090729, GO:0035821]",[GO:0035821],[GO:0090729],None,"[GO:0044419, GO:0090729, GO:0008150, GO:000367...","[GO:0044419, GO:0008150, GO:0035821]","[GO:0090729, GO:0003674]",None
417093,A1YEV9,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Gorilla gorilla gorilla,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872]","[GO:0005634, GO:0005634]","[GO:0071840, GO:0009987, GO:0008150, GO:003250...","[GO:0007283, GO:0048869, GO:0003006, GO:007184...","[GO:0043169, GO:0043167, GO:0003677, GO:000367...","[GO:0110165, GO:0043227, GO:0005634, GO:004323..."
417094,A1YG26,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Pan paniscus,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872, GO:0000978, GO:0001228]","[GO:0005634, GO:0005634]","[GO:1902680, GO:0071840, GO:0001216, GO:004589...","[GO:1902680, GO:0071840, GO:0048518, GO:000998...","[GO:0003690, GO:0001216, GO:0001228, GO:009715...","[GO:0110165, GO:0043227, GO:0005634, GO:004323..."
417095,A2T712,Zinc finger and SCAN domain-containing protein 21,Strong transcriptional activator (By similarit...,Pan troglodytes,473,Nucleus,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"[GO:0007141, GO:0007283, GO:0003677, GO:005132...","[GO:0007141, GO:0007283, GO:0051321, GO:003015...","[GO:0003677, GO:0046872, GO:0000978, GO:0001228]","[GO:0005634, GO:0005634]","[GO:1902680, GO:0071840, GO:0001216, GO:004589...","[GO:1902680, GO:0071840, GO:0048518, GO:000998...","[GO:0003690, GO:0001216, GO:0001228, GO:009715...","[GO:0110165, GO:0043227, GO:0005634, GO:004323..."


In [36]:
swissprot_data.iloc[417092]

protein_id                                                         P38442
protein_names                                        Zona occludens toxin
protein_function        Increases the permeability of the small intest...
organism                Vibrio cholerae serotype O1 (strain ATCC 39315...
length                                                                399
subcellular_location                                                 None
sequence                MSIFIHHGAPGSYKTSGALWLRLLPAIKSGRHIITNVRGLNLERMA...
go_ids                                           [GO:0090729, GO:0035821]
go_bp                                                        [GO:0035821]
go_mf                                                        [GO:0090729]
go_cc                                                                None
go_ids_extended         [GO:0044419, GO:0090729, GO:0008150, GO:000367...
go_bp_extended                       [GO:0044419, GO:0008150, GO:0035821]
go_mf_extended                        

In [37]:
swissprot_data.iloc[417092]['go_ids_extended']

['GO:0044419', 'GO:0090729', 'GO:0008150', 'GO:0003674', 'GO:0035821']

In [38]:
swissprot_data = swissprot_data.drop(columns=['go_ids', 'go_bp', 'go_mf', 'go_cc'])
swissprot_data = swissprot_data.rename(columns={'go_ids_extended':'go_ids', 'go_bp_extended':'go_bp',
                                                'go_mf_extended':'go_mf', 'go_cc_extended': 'go_cc'})

## Checking GO Categories

In [39]:
from tqdm import tqdm
from goatools.obo_parser import GODag
import pandas as pd

# Load ontology
obodag = GODag("go-basic.obo")

# Map GO term -> namespace
go_to_aspect = {go_id: term.namespace for go_id, term in obodag.items()}

# Expected aspect mapping for your columns
col_to_aspect = {
    "go_bp": "biological_process",
    "go_mf": "molecular_function",
    "go_cc": "cellular_component"
}

# Track invalid assignments
invalid_assignments = {col: set() for col in col_to_aspect.keys()}
proteins_with_misassigned = 0

def check_aspect(go_terms, expected_aspect, colname):
    global proteins_with_misassigned
    if not isinstance(go_terms, list) or not go_terms:
        return None
    
    wrong = [go for go in go_terms if go in go_to_aspect and go_to_aspect[go] != expected_aspect]
    if wrong:
        proteins_with_misassigned += 1
        invalid_assignments[colname].update(wrong)
    return wrong if wrong else None

# Apply to dataframe
tqdm.pandas(desc="Checking aspect consistency")
for col, expected in col_to_aspect.items():
    swissprot_data[col].progress_apply(
        lambda terms: check_aspect(terms, expected, col)
    )

# Summary
for col in col_to_aspect:
    print(f"{col}: {len(invalid_assignments[col])} unique misassigned GO terms")

print(f"Proteins with at least one misassigned GO term: {proteins_with_misassigned}")

go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms


Checking aspect consistency: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:00<00:00, 758608.50it/s]

go_bp: 0 unique misassigned GO terms
go_mf: 0 unique misassigned GO terms
go_cc: 0 unique misassigned GO terms
Proteins with at least one misassigned GO term: 0


**Sorting to ensure that the go columns are sorted**

In [40]:
from goatools.obo_parser import GODag

# Load the ontology file (.obo). You can download it from http://purl.obolibrary.org/obo/go.obo
obo_file = "go-basic.obo"
go_dag = GODag(obo_file)

# Build dictionary mapping GO term → depth
go_to_depth = {go_id: term.depth for go_id, term in go_dag.items()}

# List of GO columns to sort
go_columns = ["go_ids", "go_bp", "go_mf", "go_cc"]

# Function to sort a list of GO terms by depth
def sort_by_depth(go_list, go_to_depth):
    if isinstance(go_list, list) and go_list:
        # sort ascending by depth (shallowest first)
        return sorted(go_list, key=lambda go: go_to_depth[go])
    return go_list  # keep None or empty as-is

# Apply to all columns with a progress bar
from tqdm import tqdm
tqdm.pandas(desc="Sorting GO term columns")

for col in go_columns:
    swissprot_data[col] = swissprot_data[col].progress_apply(lambda x: sort_by_depth(x, go_to_depth))

go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms


Sorting GO term columns: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:00<00:00, 710835.81it/s]


In [41]:
import pandas as pd
import numpy as np

def validate_go_columns_fast(df, columns, go_to_depth):
    results = {}
    depth_lookup = go_to_depth  # local copy

    for col in columns:
        def check_sorted(go_list):
            # skip empty or invalid rows
            if not isinstance(go_list, (list, np.ndarray)) or not go_list:
                return True
            depths = [depth_lookup[go] for go in go_list]
            return all(earlier <= later for earlier, later in zip(depths, depths[1:]))

        results[col] = df[col].apply(check_sorted).all()
    return results

columns_to_check = ["go_ids", "go_bp", "go_mf", "go_cc"]
validation_results = validate_go_columns_fast(swissprot_data, columns_to_check, go_to_depth)

print("All GO term columns sorted by depth:")
for col, is_valid in validation_results.items():
    print(f"{col}: {is_valid}")

All GO term columns sorted by depth:
go_ids: True
go_bp: True
go_mf: True
go_cc: True


In [42]:
from tqdm import tqdm
tqdm.pandas(desc="Sanity check: GO categories sum to go_ids")

def check_go_sum(row):
    go_ids = set(row["go_ids"]) if isinstance(row["go_ids"], list) else set()
    go_union = set()
    for col in ["go_bp", "go_mf", "go_cc"]:
        if isinstance(row[col], list):
            go_union.update(row[col])
    return go_ids == go_union

# Apply to all rows
sanity_check_passed = swissprot_data.progress_apply(check_go_sum, axis=1).all()

print(f"Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? {sanity_check_passed}")

# Optional: if you want to see which rows fail
failed_rows = swissprot_data[~swissprot_data.progress_apply(check_go_sum, axis=1)]
print(f"Number of rows failing sanity check: {len(failed_rows)}")

Sanity check: GO categories sum to go_ids: 100%|████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:07<00:00, 53052.38it/s]


Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? True


Sanity check: GO categories sum to go_ids: 100%|████████████████████████████████████████████████████████████████████████████████████████| 417097/417097 [00:07<00:00, 54617.32it/s]

Number of rows failing sanity check: 0


In [44]:
from sklearn.model_selection import train_test_split

# 80:20 split
train_df, test_df = train_test_split(swissprot_data, test_size=0.2, random_state=42, shuffle=True)

print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 333677
Test set size: 83420


In [45]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(train_df, preserve_index=False)
test_hf = Dataset.from_pandas(test_df, preserve_index=False)

In [46]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [47]:
swissprot_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'],
        num_rows: 333677
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'],
        num_rows: 83420
    })
})

In [48]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="swissprot_reasoning",
    commit_message="Big Change: Removed CAFA5 proteins and added in GO Terms from GOA and not just uniprot"
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/95.7M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/95.8M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/84 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/b1a3dafe2bc5ca32ff5aceaaf9d36a9d7c586dd9', commit_message='Big Change: Removed CAFA5 proteins and added in GO Terms from GOA and not just uniprot', commit_description='', oid='b1a3dafe2bc5ca32ff5aceaaf9d36a9d7c586dd9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Download AlphaFold Structures for all Proteins

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [1]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")

In [2]:
protein_ids = set(list(ds['train']['protein_id']) + list(ds['test']['protein_id']))
len(protein_ids)

417097

In [3]:
print(f"Number of proteins that overlap between test and train set: {len(list(set(list(ds['train']['protein_id'])) & set(list(set(list(ds['test']['protein_id']))))))}")

Number of proteins that overlap between test and train set: 0


### Downloading AlphaFoldDB

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [4]:
with open("alphafold_swissprot_manifest.txt", "w") as f:
    for protein in (protein_ids):
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{protein}-F1-model_v4.cif\n")

```cat alphafold_swissprot_manifest.txt | gsutil -m cp -I af_db_swissprot/```

In [8]:
import os

# Combined list of all expected accessions
all_expected = protein_ids

# Get all filenames in af_db/ and extract accessions
downloaded_files = os.listdir("af_db_swissprot")
downloaded_accessions = set(
    filename.split("-")[1]  # Extract accession from 'AF-<ACCESSION>-F1-model_v4.cif'
    for filename in downloaded_files
    if filename.endswith(".cif")
)

# Find accessions that are missing
missing_accessions = sorted(all_expected - downloaded_accessions)

print(f"Expected: {len(all_expected)}")
print(f"Downloaded: {len(downloaded_accessions)}")
print(f"Missing: {len(missing_accessions)}")

# Save to file
with open("missing_af_accessions.txt", "w") as f:
    for acc in missing_accessions:
        f.write(f"{acc}\n")

Expected: 417097
Downloaded: 396393
Missing: 20704


Switch to Bash terminal

In [ ]:
mkdir -p af_shards

i=0
shard=0

for file in af_db_swissprot/*.cif; do
    # Create a new folder every 5000 files
    if (( i % 5000 == 0 )); then
        shard_dir="af_shards/shard_${shard}"
        mkdir -p "$shard_dir"
        ((shard++))
    fi

    cp "$file" "$shard_dir/"
    ((i++))
done

In [ ]:
cd af_shards

ls -d shard_* | parallel --bar 'tar -czf {}.tar.gz {} && rm -r {}'

cd ..

In [ ]:

git lfs install
git clone https://huggingface.co/datasets/wanglab/cafa5

mkdir cafa5/swissprot_structures
cp -r af_shards/ cafa5/swissprot_structures/


git add swissprot_structures/
git commit -m "Add AlphaFold structures for swissprot proteins"
git push

# Adding Structure Path column

In [1]:
import pandas as pd

In [2]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")

Generating train split:   0%|          | 0/333677 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/83420 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 333677
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 83420
    })
})

In [4]:
swissprot_train = ds['train'].to_pandas()
swissprot_test = ds['test'].to_pandas()

In [5]:
with open("missing_af_accessions.txt", "r") as f:
    missing_accessions = [line.strip() for line in f if line.strip()]

In [6]:
len(missing_accessions)

20704

In [7]:
swissprot_train['structure_path'] = swissprot_train['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v4.cif" if x not in missing_accessions else None
)
swissprot_test['structure_path'] = swissprot_test['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v4.cif" if x not in missing_accessions else None
)

In [8]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(swissprot_train, preserve_index=False)
test_hf = Dataset.from_pandas(swissprot_test, preserve_index=False)

In [9]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [10]:
swissprot_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 333677
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 83420
    })
})

In [11]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="swissprot_reasoning",
    commit_message="Added structure paths and structures to swissprot data"
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/84 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/80e0f9915e6ad5e30ef11be630489bc395968334', commit_message='Added structure paths and structures to swissprot data', commit_description='', oid='80e0f9915e6ad5e30ef11be630489bc395968334', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Getting InterPro Domains for the Proteins

In [1]:
import pandas as pd

In [2]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 333677
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path'],
        num_rows: 83420
    })
})

In [4]:
swissprot_train = ds['train'].to_pandas()
swissprot_test = ds['test'].to_pandas()

In [ ]:
from datasets import load_dataset
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from tqdm import tqdm

def fetch_one_protein(accession, max_retries=3):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/InterPro/protein/UniProt/{accession}"
    headers = {"Accept": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 204:
                return []
            r.raise_for_status()

            results = r.json().get("results", [])
            rows = []

            for item in results:
                ipr_id = item.get("metadata", {}).get("accession")
                entry_name = item.get("metadata", {}).get("name")
                ipr_type = item.get("metadata", {}).get("type")

                all_fragments = []
                for loc in item.get("proteins", []):  # fixed: must go through item['proteins']
                    for entry in loc.get("entry_protein_locations", []):
                        all_fragments.extend(entry.get("fragments", []))

                if all_fragments:
                    start = min(f["start"] for f in all_fragments if "start" in f)
                    end = max(f["end"] for f in all_fragments if "end" in f)

                    rows.append({
                        "accession": accession,
                        "interpro_id": ipr_id,
                        "entry_name": entry_name,
                        "type": ipr_type,
                        "start": start,
                        "end": end,
                        "n_fragments": len(all_fragments)
                    })
            return rows

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Failed for {accession}: {e}")
            sleep(2 * (attempt + 1))
    return []

def fetch_all_interpro_parallel(
    all_accessions,
    output_file="interpro_domains.tsv",
    interpro_lookup_file="interpro_lookup.tsv",
    max_threads=32
):
    protein_rows = []
    interpro_lookup = {}

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(fetch_one_protein, acc): acc for acc in all_accessions}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching InterPro"):
            rows = future.result()
            protein_rows.extend(rows)

    # Extract lookup table
    for row in protein_rows:
        ipr = row["interpro_id"]
        if ipr not in interpro_lookup:
            interpro_lookup[ipr] = {
                "entry_name": row["entry_name"],
                "type": row["type"]
            }

    # Save data
    pd.DataFrame(protein_rows).to_csv(output_file, sep="\t", index=False)
    pd.DataFrame([
        {"interpro_id": ipr, "entry_name": val["entry_name"], "type": val["type"]}
        for ipr, val in interpro_lookup.items()
    ]).to_csv(interpro_lookup_file, sep="\t", index=False)

    print(f"✅ Done. Fetched {len(protein_rows)} entries across {len(all_accessions)} proteins")

# ---------- Main execution ----------
if __name__ == "__main__":
    ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")
    swissprot_train = ds['train'].to_pandas()
    swissprot_test = ds['test'].to_pandas()

    proteins = set(list(swissprot_train['protein_id']) + list(swissprot_test['protein_id']))

    print(f"Fetched {len(proteins)} proteins")

    fetch_all_interpro_parallel(
        list(proteins),
        output_file="swissprot_interpro_domains.tsv",
        interpro_lookup_file="swissprot_interpro_lookup.tsv",
        max_threads=32
    )

In [5]:
interpro = pd.read_csv('swissprot_interpro_domains.tsv', sep='\t')

In [6]:
interpro

,accession,interpro_id,entry_name,type,start,end,n_fragments
0,Q3J366,IPR000119,Histone-like DNA-binding protein,family,4,95,1
1,Q3J366,IPR005684,"Integration host factor, alpha subunit",family,4,100,1
2,Q3J366,IPR010992,Integration host factor (IHF)-like DNA-binding...,homologous_superfamily,1,96,1
3,Q3J366,IPR020816,"Histone-like DNA-binding protein, conserved site",conserved_site,51,70,1
4,A1JS33,IPR002222,Small ribosomal subunit protein uS19,family,1,92,1
...,...,...,...,...,...,...,...
1919605,Q890R2,IPR015856,"ABC transporter, CbiO/EcfA subunit",domain,7,222,1
1919606,Q890R2,IPR017871,"ABC transporter-like, conserved site",conserved_site,146,160,1
1919607,Q890R2,IPR027417,P-loop containing nucleoside triphosphate hydr...,homologous_superfamily,2,281,1
1919608,Q890R2,IPR030947,Energy-coupling factor transporter ATP-binding...,family,6,278,1


In [7]:
set(interpro['type'])

{'active_site',
 'binding_site',
 'conserved_site',
 'domain',
 'family',
 'homologous_superfamily',
 'ptm',
 'repeat'}

In [8]:
from collections import defaultdict

# Define your custom type priority
type_order = {
    'homologous_superfamily': 0,
    'family': 1,
    'domain': 2,
    'conserved_site': 3,
    'ptm': 4,
    'repeat': 5,
    'active_site': 6,
    'binding_site': 7
}

# Intermediate mapping
accession_to_entries = defaultdict(list)

# Store everything per accession
for acc, ipr, start, end, typ in zip(
    interpro["accession"], interpro["interpro_id"],
    interpro["start"], interpro["end"], interpro["type"]
):
    accession_to_entries[acc].append((ipr, start, end, typ))

# Final output structure
collapsed_data = []

for acc, entries in accession_to_entries.items():
    interpro_ids = set()
    interpro_location = {}
    
    # Sort by (type priority, start)
    entries_sorted = sorted(
        entries,
        key=lambda x: (type_order.get(x[3], 99), x[1])  # (type, start)
    )

    for ipr, start, end, typ in entries_sorted:
        if ipr in interpro_location:
            raise ValueError(f"InterPro ID {ipr} appears more than once for protein {acc}")
        interpro_ids.add(ipr)
        interpro_location[ipr] = (start, end)

    collapsed_data.append({
        "accession": acc,
        "interpro_ids": list(interpro_ids),
        "interpro_location": interpro_location
    })

# Convert to DataFrame
collapsed_df = pd.DataFrame(collapsed_data)

In [9]:
collapsed_df

,accession,interpro_ids,interpro_location
0,Q3J366,"[IPR005684, IPR010992, IPR000119, IPR020816]","{'IPR010992': (1, 96), 'IPR000119': (4, 95), '..."
1,A1JS33,"[IPR002222, IPR005732, IPR023575, IPR020934]","{'IPR023575': (1, 92), 'IPR002222': (1, 92), '..."
2,Q13X06,"[IPR013785, IPR006269, IPR006218]","{'IPR013785': (1, 282), 'IPR006269': (3, 277),..."
3,Q981B9,"[IPR005862, IPR003135, IPR048740, IPR016185, I...","{'IPR016185': (3, 111), 'IPR013815': (123, 195..."
4,P0A3G1,"[IPR047870, IPR050530, IPR018493, IPR000638]","{'IPR047870': (4, 71), 'IPR050530': (5, 60), '..."
...,...,...,...
406154,C6DAX0,"[IPR036291, IPR020631, IPR046346, IPR020867, I...","{'IPR046346': (1, 122), 'IPR036291': (123, 283..."
406155,Q04YY5,"[IPR041445, IPR036388, IPR027417, IPR036390, I...","{'IPR027417': (19, 249), 'IPR036390': (255, 32..."
406156,P0A8Q7,"[IPR003769, IPR022935, IPR014719]","{'IPR014719': (1, 106), 'IPR022935': (14, 105)..."
406157,Q9P9J4,"[IPR022667, IPR002770, IPR023447, IPR014053]","{'IPR023447': (1, 295), 'IPR014053': (1, 296),..."


# Update Train and Test with Interpro Data

In [10]:
interpro_collapsed = collapsed_df.rename(columns={"accession": "protein_id"})

swissprot_train = swissprot_train.merge(interpro_collapsed, on="protein_id", how="left")
swissprot_test = swissprot_test.merge(interpro_collapsed, on="protein_id", how="left")

In [11]:
print("Rows where train interpro is None:", swissprot_train["interpro_ids"].isnull().sum())
print("Rows where test interpro is None:", swissprot_test["interpro_ids"].isnull().sum())

Rows where train interpro is None: 8711
Rows where test interpro is None: 2227


## Interpro Metadata

In [12]:
interpro_metadata = pd.read_csv('swissprot_interpro_lookup.tsv', sep='\t')

In [13]:
interpro_metadata

,interpro_id,entry_name,type
0,IPR000119,Histone-like DNA-binding protein,family
1,IPR005684,"Integration host factor, alpha subunit",family
2,IPR010992,Integration host factor (IHF)-like DNA-binding...,homologous_superfamily
3,IPR020816,"Histone-like DNA-binding protein, conserved site",conserved_site
4,IPR002222,Small ribosomal subunit protein uS19,family
...,...,...,...
25268,IPR055187,"BIRD-IDD transcription factor, third C2HC zinc...",domain
25269,IPR053502,Magnetosome CDF Transporter-Related,family
25270,IPR031655,"FokI, D3 domain",domain
25271,IPR016874,Methyltransferase TcmP-like,family


**QC to check if every interpro ID in swissprot reasoning is in this metadata df**

In [14]:
# Step 1: create a set of valid InterPro IDs
valid_ids = set(interpro_metadata["interpro_id"].dropna())

# Step 2: explode interpro_ids into separate rows
exploded = swissprot_train[["interpro_ids"]].explode("interpro_ids")

# Step 3: filter out null values first
exploded = exploded[exploded["interpro_ids"].notna()]

# Step 4: mark invalid IDs
exploded["is_invalid"] = ~exploded["interpro_ids"].isin(valid_ids)

# Step 5: count invalids
invalid_count = exploded["is_invalid"].sum()

print(invalid_count if invalid_count > 0 else 0)

0


In [15]:
# Step 1: create a set of valid InterPro IDs
valid_ids = set(interpro_metadata["interpro_id"].dropna())

# Step 2: explode interpro_ids into separate rows
exploded = swissprot_test[["interpro_ids"]].explode("interpro_ids")

# Step 3: filter out null values first
exploded = exploded[exploded["interpro_ids"].notna()]

# Step 4: mark invalid IDs
exploded["is_invalid"] = ~exploded["interpro_ids"].isin(valid_ids)

# Step 5: count invalids
invalid_count = exploded["is_invalid"].sum()

print(invalid_count if invalid_count > 0 else 0)

0


**All interpro ids in swissprot dataframes are in this metadata df**

# Upload Data

In [17]:
import json

In [18]:
#Making it into a json object so that huggingface can handle it
swissprot_train["interpro_location"] = swissprot_train["interpro_location"].apply(json.dumps)
swissprot_test["interpro_location"] = swissprot_test["interpro_location"].apply(json.dumps)

In [19]:
from datasets import Dataset

train_hf = Dataset.from_pandas(swissprot_train, preserve_index=False)
test_hf = Dataset.from_pandas(swissprot_test, preserve_index=False)

In [20]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [21]:
swissprot_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location'],
        num_rows: 333677
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location'],
        num_rows: 83420
    })
})

In [22]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="swissprot_reasoning",
    commit_message="Uploaded InterPro domains with all metadata, including start and end and type"
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/110M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/167 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/110M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/84 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/54.8M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/82cb0e519a82d429befd3bcea61591bfff679cc1', commit_message='Uploaded InterPro domains with all metadata, including start and end and type', commit_description='', oid='82cb0e519a82d429befd3bcea61591bfff679cc1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [23]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
interpro_hf = Dataset.from_pandas(interpro_metadata, preserve_index=False)

In [24]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "metadata": interpro_hf,
})

In [25]:
swissprot_dataset

DatasetDict({
    metadata: Dataset({
        features: ['interpro_id', 'entry_name', 'type'],
        num_rows: 25273
    })
})

In [27]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interpro_swissprot_metadata",
    commit_message="Upload the InterPro Swissprot Metadata, downloaded from the Uniprot Cross references"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/26 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/729k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/1a1ce206756de3cca3196ed78e10761b1d7ca3e1', commit_message='Upload the InterPro Swissprot Metadata, downloaded from the Uniprot Cross references', commit_description='', oid='1a1ce206756de3cca3196ed78e10761b1d7ca3e1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Cleaning up + Making 95-5 Split

In [1]:
import pandas as pd

In [2]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'full_interaction_info'],
        num_rows: 378471
    })
    val: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'full_interaction_info'],
        num_rows: 19920
    })
})

In [4]:
swissprot_train = ds['train'].to_pandas()
swissprot_val = ds['val'].to_pandas()

In [5]:
swissprot_all = pd.concat([swissprot_train,swissprot_val], ignore_index = True)

In [16]:
# Get duplicated rows based on 'protein_id'
duplicates = swissprot_all[swissprot_all['protein_id'].duplicated(keep=False)]

# Filter where string_id contains 'gene'
duplicates_with_gene = duplicates[duplicates['string_id'].str.contains('gene', case=False, na=False)]

# Drop these rows from swissprot_all
swissprot_all = swissprot_all.drop(duplicates_with_gene.index)

In [17]:
swissprot_all = swissprot_all[~swissprot_all['go_ids'].isna()].reset_index(drop=True)

In [18]:
import numpy as np

swissprot_all['interpro_location'] = swissprot_all['interpro_location'].replace("NaN", np.nan)

In [19]:
from sklearn.model_selection import train_test_split

# 95:5 split
train_df, val_df = train_test_split(swissprot_all, test_size=0.05, random_state=42, shuffle=True)

print(f"Train set size: {len(train_df)}")
print(f"Val set size: {len(val_df)}")

Train set size: 376749
Val set size: 19829


In [20]:
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df, preserve_index=False)
val_hf = Dataset.from_pandas(val_df, preserve_index=False)

In [21]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "train": train_hf,
    "val": val_hf
})

In [22]:
swissprot_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'full_interaction_info'],
        num_rows: 376749
    })
    val: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'full_interaction_info'],
        num_rows: 19829
    })
})

In [23]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="swissprot_reasoning",
    commit_message="Clean up the Data a bit"
)

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/126 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/125M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/126 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/125M [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/125M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/f3b5b3fdf794ff29507bd8a1dd07386dba3d43d0', commit_message='Clean up the Data a bit', commit_description='', oid='f3b5b3fdf794ff29507bd8a1dd07386dba3d43d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)